# ARC-GEN と ARC-AGI 1 の分布比較

この notebook は、`kaggle/combined/arc-agi_training-*.json` にある ARC-AGI 1 の元データと、`data/arc-gen/*.json` にある ARC-GEN の生成例を比較します。

比較する観点:
- データ量と 1 タスクあたりペア数
- 入出力サイズの分布
- 面積・色数の分布
- 0 色比率・エントロピー・遷移率などの構造指標
- 入出力変換の傾向
- 色使用傾向
- タスク単位で見た平均傾向のズレ

日本語フォントが環境に無い場合は、最初の実行時に `Noto Sans CJK JP` を `.cache/fonts` にダウンロードします。

In [ ]:
from pprint import pprint

import matplotlib.pyplot as plt

from data.arc_distribution_compare import (
    configure_japanese_font,
    dataset_overview,
    discover_repo_root,
    plot_color_usage,
    plot_overview,
    plot_pair_transform_distributions,
    plot_shape_change_breakdown,
    plot_shape_heatmaps,
    plot_size_palette_distributions,
    plot_structure_detail_distributions,
    plot_structure_distributions,
    plot_task_level_scatter,
    plot_top_task_gap_bars,
    prepare_comparison_dataset,
    standardized_mean_differences,
)

REPO_ROOT = discover_repo_root()
ORIGINAL_SCOPE = "all"  # "all" なら train + test, "train" なら train のみ

FONT_NAME = configure_japanese_font(REPO_ROOT / ".cache" / "fonts")
comparison = prepare_comparison_dataset(REPO_ROOT, original_scope=ORIGINAL_SCOPE)
example_records = comparison["example_records"]
task_records = comparison["task_records"]
overview = dataset_overview(example_records, task_records)

print(f"repo_root: {REPO_ROOT}")
print(f"日本語フォント: {FONT_NAME}")
print(f"original_scope: {ORIGINAL_SCOPE}")
print(f"比較対象タスク数: {len(comparison['task_ids'])}")
print(f"ARC-AGI 1 ペア数: {sum(1 for row in example_records if row['source'] == 'ARC-AGI 1')}")
print(f"ARC-GEN ペア数:   {sum(1 for row in example_records if row['source'] == 'ARC-GEN')}")

In [ ]:
print("概要統計")
pprint(overview)

In [ ]:
comparison_fields = [
    "input_area",
    "output_area",
    "input_palette_size",
    "output_palette_size",
    "input_zero_fraction",
    "output_zero_fraction",
    "input_entropy_bits",
    "output_entropy_bits",
    "input_horizontal_transition_rate",
    "input_vertical_transition_rate",
    "input_unique_rows_ratio",
    "input_unique_cols_ratio",
    "input_horizontal_symmetry",
    "input_vertical_symmetry",
    "area_ratio",
    "overlap_change_rate",
    "palette_jaccard",
    "introduced_colors",
    "removed_colors",
]

mean_gap_rows = standardized_mean_differences(example_records, comparison_fields)
print("標準化平均との差が大きい指標 上位 12 件")
for row in mean_gap_rows[:12]:
    print(
        f"{row['metric']:<32} "
        f"ARC-AGI 1={row['arc_agi_mean']:.4f}  "
        f"ARC-GEN={row['arc_gen_mean']:.4f}  "
        f"Δ={row['delta']:.4f}  "
        f"z差={row['standardized_delta']:.4f}"
    )

In [ ]:
fig = plot_overview(example_records, task_records)
plt.show()

In [ ]:
fig = plot_shape_heatmaps(example_records)
plt.show()

In [ ]:
fig = plot_size_palette_distributions(example_records)
plt.show()

In [ ]:
fig = plot_structure_distributions(example_records)
plt.show()

fig = plot_structure_detail_distributions(example_records)
plt.show()

In [ ]:
fig = plot_pair_transform_distributions(example_records)
plt.show()

fig = plot_shape_change_breakdown(example_records)
plt.show()

In [ ]:
fig = plot_color_usage(example_records)
plt.show()

In [ ]:
fig = plot_task_level_scatter(task_records)
plt.show()

fig = plot_top_task_gap_bars(task_records)
plt.show()

## 読み方のヒント

- ヒストグラムは密度で正規化しているので、データ数の差が大きくても形の違いを比較しやすくしています。
- ヒートマップは高さ × 幅の出現頻度です。色が強いほど、そのサイズのペアが多いことを表します。
- `標準化平均との差` は、単位の違う指標を横並びで見るための簡易スコアです。絶対値が大きいほど、ARC-GEN と ARC-AGI 1 の差が目立つ指標です。
- タスク対応散布図は、各 task id を 1 点として、ARC-AGI 1 と ARC-GEN のタスク平均を比較しています。対角線から離れるほど、そのタスクで分布差が大きいことを意味します。